In [ ]:
pip install pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report



In [ ]:
rows = 10000
np.random.seed(42)

data = []

for i in range(rows):

    # 🔹 Simulate user state first
    state = np.random.choice(["Low", "Medium", "High"], p=[0.4, 0.35, 0.25])

    if state == "Low":
        typing_speed = np.random.normal(55, 5)
        mouse_speed = np.random.normal(850, 100)
        click_rate = np.random.normal(22, 4)
        idle_time = np.random.normal(8, 4)
        app_switches = np.random.normal(4, 2)

    elif state == "Medium":
        typing_speed = np.random.normal(42, 6)
        mouse_speed = np.random.normal(650, 120)
        click_rate = np.random.normal(16, 4)
        idle_time = np.random.normal(20, 8)
        app_switches = np.random.normal(7, 3)

    else:  # High load
        typing_speed = np.random.normal(30, 6)
        mouse_speed = np.random.normal(450, 120)
        click_rate = np.random.normal(10, 3)
        idle_time = np.random.normal(45, 10)
        app_switches = np.random.normal(12, 4)

    # 🔹 Add random noise (important!)
    typing_speed += np.random.normal(0, 2)
    mouse_speed += np.random.normal(0, 30)
    click_rate += np.random.normal(0, 1.5)
    idle_time += np.random.normal(0, 3)
    app_switches += np.random.normal(0, 1)

    # 🔹 Clip values (real-world limits)
    typing_speed = np.clip(typing_speed, 10, 80)
    mouse_speed = np.clip(mouse_speed, 100, 1200)
    click_rate = np.clip(click_rate, 1, 40)
    idle_time = np.clip(idle_time, 0, 120)
    app_switches = np.clip(app_switches, 0, 25)

    data.append([
        typing_speed,
        mouse_speed,
        click_rate,
        idle_time,
        app_switches,
        state
    ])

df = pd.DataFrame(data, columns=[
    "typing_speed",
    "mouse_speed",
    "click_rate",
    "idle_time",
    "app_switches",
    "label"
])

# 🔹 Shuffle dataset
df = df.sample(frac=1).reset_index(drop=True)

# Save
df.to_csv("dataset.csv", index=False)

print("✅ Realistic dataset created: dataset.csv")

In [ ]:
# =========================================
# LOAD DATA
# =========================================

df = pd.read_csv("dataset.csv")

print("\n📂 Dataset Loaded:")
print(df.head())

# =========================================
# PREPARE DATA
# =========================================

X = df.drop("label", axis=1)
y = df["label"]

# Normalize
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:


# =========================================
# TRAIN MODEL
# =========================================

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
pred = model.predict(X_test)

print("\n📊 Model Performance:")
print(classification_report(y_test, pred))


In [ ]:
from datetime import datetime, timedelta

tasks = []

PRIORITY_WEIGHT = 0.6
DEADLINE_WEIGHT = 0.4

# =========================================
# AUTO DURATION FUNCTION
# =========================================

def estimate_duration(priority, deadline_hours):
    base = 2 + (priority * 4)
    urgency_factor = 1 / (1 + deadline_hours)
    duration = base * (1 - 0.5 * urgency_factor)
    return round(max(0.5, duration), 2)

# =========================================
# INPUT
# =========================================

n = int(input("Enter number of tasks: "))

for i in range(n):
    print(f"\nTask {i+1}")

    name = input("Task name: ")
    priority = float(input("Priority (0 to 1): "))
    deadline_hours = float(input("Deadline (in hours): "))

    duration = estimate_duration(priority, deadline_hours)

    tasks.append({
        "name": name,
        "priority": priority,
        "deadline_hours": deadline_hours,
        "duration": duration
    })

# =========================================
# SCORE CALCULATION
# =========================================

for task in tasks:
    urgency = 1 / (1 + task["deadline_hours"])

    task["score"] = (
        PRIORITY_WEIGHT * task["priority"] +
        DEADLINE_WEIGHT * urgency
    )

# =========================================
# SORT BY SCORE (internal)
# =========================================

tasks_sorted = sorted(tasks, key=lambda x: x["score"], reverse=True)

# =========================================
# BUILD TIMELINE
# =========================================

current_time = datetime.now()

for t in tasks_sorted:
    t["start"] = current_time
    t["end"] = current_time + timedelta(hours=t["duration"])
    current_time = t["end"]

# =========================================
# SORT BY START TIME (FOR OUTPUT)
# =========================================

timeline = sorted(tasks_sorted, key=lambda x: x["start"])

# =========================================
# DISPLAY
# =========================================

print("\n📅 Your Task Timeline (Follow this order):\n")

for t in timeline:
    print(f"{t['start'].strftime('%H:%M')} → {t['end'].strftime('%H:%M')}")
    print(f"  Task: {t['name']}")
    print(f"  Priority: {t['priority']}")
    print(f"  ⏱ Duration: {t['duration']} hrs")
    print("-" * 40)